# Millog 피로도 AI — ML 학습 노트북 v1

> **목적**: 규칙 기반 피로도 엔진을 보강하는 Ridge Regression 모델 학습  
> **데이터**: `fatigue_logs.jsonl` (실제 서비스에서 수집된 피로도 계산 기록)  
> **출력**: `fatigue_ml_model.joblib` → `millog-fatigue/models/` 에 저장  

## 실행 조건
- Phase 2: 최소 500건 이상의 `fatigue_logs.jsonl` 데이터 필요
- Phase 1(현재): 합성 데이터로 파이프라인 검증 가능

## 피로도 공식 (규칙 기반, 참고)
```
F = clamp( Σ(Di × Ti × hi) × C + P - R, 0, 10 )
Di: 근무 유형별 피로지수, Ti: 시간대 가중치, hi: 근무 시간
C: 연속근무 계수, P: 수면 중단 페널티, R: 수면 회복량
```

## 0. 환경 설정

In [ ]:
# Google Colab에서 실행 시 패키지 설치
!pip install scikit-learn pandas numpy matplotlib seaborn joblib --quiet

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import os

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ 환경 설정 완료')

## 1. 데이터 로드

### 실제 데이터 사용 시
Render 서버에서 `fatigue_logs.jsonl`을 다운로드하여 업로드하세요:
```bash
# 서버에서 다운로드
curl -o fatigue_logs.jsonl https://your-api.onrender.com/fatigue/logs/export
```

### Phase 1: 합성 데이터 (파이프라인 검증용)
실제 데이터가 없을 때 규칙 기반 엔진으로 합성 데이터를 생성합니다.

In [ ]:
# 규칙 기반 상수 (millog-fatigue/fatigue/constants.py 동기화)
DUTY_FATIGUE_INDEX = {
    'gop_gp': 2.63,
    'guard_duty': 2.45,
    'field_training': 2.33,
    'guard_post': 2.25,
    'night_watch': 2.16,
    'cctv_surveillance': 2.00,
    'indoor_training': 0.80,
    'admin_work': 0.50,
    'none': 0.00,
}

TIME_WEIGHT = {'day': 0.80, 'evening': 0.90, 'night': 1.00, 'dawn': 1.15}
SLEEP_QUALITY = {'continuous_6plus': 1.00, 'split': 0.53, 'under_4': 0.35}

def rule_based_fatigue(duty_type, duty_hours, time_zone, consecutive_days,
                       sleep_hours, sleep_type, interruptions):
    """규칙 기반 피로도 계산 (합성 데이터 생성용)"""
    di = DUTY_FATIGUE_INDEX.get(duty_type, 0.0)
    ti = TIME_WEIGHT.get(time_zone, 0.80)
    c = min(1.0 + 0.15 * (consecutive_days - 1), 2.0)
    p = interruptions * 3.0
    quality = SLEEP_QUALITY.get(sleep_type, 0.53)
    r = sleep_hours * quality
    raw = (di * ti * duty_hours) * c + p - r
    return float(np.clip(raw, 0, 10))

# 합성 데이터 생성
np.random.seed(42)
N = 1000  # 실제 데이터가 없을 때 합성

duty_types = list(DUTY_FATIGUE_INDEX.keys())
time_zones = list(TIME_WEIGHT.keys())
sleep_types = list(SLEEP_QUALITY.keys())
rank_groups = ['이병', '일병', '상병', '병장']  # 계급 그룹
branch_groups = ['육군', '해군', '공군', '해병대']  # 군종

rows = []
for _ in range(N):
    duty_type = np.random.choice(duty_types)
    duty_hours = np.random.uniform(1, 8)
    time_zone = np.random.choice(time_zones)
    consecutive_days = np.random.randint(1, 8)
    sleep_hours = np.random.uniform(3, 9)
    sleep_type = np.random.choice(sleep_types)
    interruptions = np.random.randint(0, 3)
    rank = np.random.choice(rank_groups)
    branch = np.random.choice(branch_groups)

    # 규칙 기반 피로도 + 개인차 노이즈 추가 (현실 반영)
    base_score = rule_based_fatigue(
        duty_type, duty_hours, time_zone,
        consecutive_days, sleep_hours, sleep_type, interruptions
    )
    # 실제 데이터에서는 사용자 자기 보고 피로도 (0~10)
    reported_score = float(np.clip(base_score + np.random.normal(0, 0.8), 0, 10))

    rows.append({
        'duty_type': duty_type,
        'duty_hours': round(duty_hours, 2),
        'time_zone': time_zone,
        'consecutive_days': consecutive_days,
        'sleep_hours': round(sleep_hours, 2),
        'sleep_type': sleep_type,
        'interruptions': interruptions,
        'rank': rank,
        'branch': branch,
        'rule_fatigue': round(base_score, 2),
        'reported_fatigue': round(reported_score, 2),  # 학습 타겟
    })

df = pd.DataFrame(rows)
print(f'✅ 데이터 로드 완료: {len(df)}건')
df.head()

## 2. 피처 엔지니어링

In [ ]:
# 파생 피처 생성
df['di'] = df['duty_type'].map(DUTY_FATIGUE_INDEX)           # 근무 유형 피로지수
df['ti'] = df['time_zone'].map(TIME_WEIGHT)                  # 시간대 가중치
df['duty_fatigue'] = df['di'] * df['ti'] * df['duty_hours']  # Di × Ti × hi
df['consecutive_factor'] = np.minimum(1.0 + 0.15 * (df['consecutive_days'] - 1), 2.0)
df['sleep_quality'] = df['sleep_type'].map(SLEEP_QUALITY)
df['sleep_recovery'] = df['sleep_hours'] * df['sleep_quality']
df['sleep_penalty'] = df['interruptions'] * 3.0

# 야간/새벽 근무 비율 (0~1)
df['is_night_dawn'] = df['time_zone'].isin(['night', 'dawn']).astype(float)

# 수면 부족 여부
df['sleep_deficit'] = (df['sleep_hours'] < 6).astype(float)

print('피처 목록:')
print(df[['duty_type', 'duty_fatigue', 'consecutive_factor', 
          'sleep_recovery', 'sleep_penalty', 'rule_fatigue', 'reported_fatigue']].describe())

## 3. 탐색적 데이터 분석 (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 피로도 분포
axes[0, 0].hist(df['reported_fatigue'], bins=30, color='#6366f1', alpha=0.8, edgecolor='white')
axes[0, 0].set_title('자기보고 피로도 분포')
axes[0, 0].set_xlabel('피로도 (0~10)')
axes[0, 0].axvline(df['reported_fatigue'].mean(), color='red', linestyle='--', label=f'평균: {df["reported_fatigue"].mean():.2f}')
axes[0, 0].legend()

# 규칙 기반 vs 자기보고 상관관계
axes[0, 1].scatter(df['rule_fatigue'], df['reported_fatigue'], alpha=0.3, color='#6366f1', s=10)
axes[0, 1].plot([0, 10], [0, 10], 'r--', label='완벽 일치 선')
corr = df['rule_fatigue'].corr(df['reported_fatigue'])
axes[0, 1].set_title(f'규칙 기반 vs 자기보고 피로도 (r={corr:.3f})')
axes[0, 1].set_xlabel('규칙 기반 피로도')
axes[0, 1].set_ylabel('자기보고 피로도')
axes[0, 1].legend()

# 근무 유형별 평균 피로도
duty_avg = df.groupby('duty_type')['reported_fatigue'].mean().sort_values(ascending=True)
duty_avg.plot(kind='barh', ax=axes[1, 0], color='#6366f1', alpha=0.8)
axes[1, 0].set_title('근무 유형별 평균 피로도')
axes[1, 0].set_xlabel('평균 피로도')

# 수면 시간별 피로도
axes[1, 1].scatter(df['sleep_hours'], df['reported_fatigue'], alpha=0.3, color='#10b981', s=10)
axes[1, 1].set_title('수면 시간 vs 피로도')
axes[1, 1].set_xlabel('수면 시간 (h)')
axes[1, 1].set_ylabel('자기보고 피로도')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA 시각화 완료')

## 4. 모델 학습 — Ridge Regression

N=500~1000 규모에서 안전하고 해석 가능한 Ridge Regression 사용.  
범주형 피처(duty_type, time_zone, sleep_type, rank, branch)는 One-Hot Encoding.

In [ ]:
# 피처·타겟 분리
CATEGORICAL_FEATURES = ['duty_type', 'time_zone', 'sleep_type', 'rank', 'branch']
NUMERICAL_FEATURES = [
    'duty_hours',
    'consecutive_days',
    'sleep_hours',
    'interruptions',
    'duty_fatigue',          # Di × Ti × hi
    'consecutive_factor',    # C
    'sleep_recovery',        # R
    'sleep_penalty',         # P
    'is_night_dawn',
    'sleep_deficit',
]

X = df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
y = df['reported_fatigue']

# 전처리 파이프라인
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERICAL_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

# RidgeCV로 최적 alpha 자동 탐색
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RidgeCV(alphas=alphas, cv=5)),
])

# 5-fold 교차 검증
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mae = -cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')
cv_r2 = cross_val_score(pipeline, X, y, cv=kf, scoring='r2')

print(f'5-Fold 교차 검증 결과:')
print(f'  MAE:  {cv_mae.mean():.4f} ± {cv_mae.std():.4f}')
print(f'  R²:   {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')

# 최종 전체 데이터로 학습
pipeline.fit(X, y)
ridge_model = pipeline.named_steps['model']
print(f'\n최적 alpha: {ridge_model.alpha_:.4f}')

## 5. 피처 중요도 분석

In [ ]:
# 피처 이름 추출
num_feature_names = NUMERICAL_FEATURES
cat_feature_names = pipeline.named_steps['preprocessor']\
    .named_transformers_['cat']\
    .get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = num_feature_names + cat_feature_names

# 계수 절댓값으로 중요도 시각화
coef = np.abs(ridge_model.coef_)
importance_df = pd.DataFrame({'feature': all_feature_names, 'importance': coef})\
    .sort_values('importance', ascending=True).tail(20)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'], color='#6366f1', alpha=0.8)
plt.title('Ridge 모델 피처 중요도 (상위 20개)')
plt.xlabel('|계수|')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 모델 평가 — 예측값 vs 실제값

In [ ]:
y_pred = pipeline.predict(X)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)
residuals = y - y_pred

print(f'전체 데이터 성능:')
print(f'  MAE: {mae:.4f}')
print(f'  R²:  {r2:.4f}')
print(f'  잔차 평균: {residuals.mean():.4f}')
print(f'  잔차 표준편차: {residuals.std():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 예측값 vs 실제값
axes[0].scatter(y, y_pred, alpha=0.3, color='#6366f1', s=10)
axes[0].plot([0, 10], [0, 10], 'r--', label='완벽 일치')
axes[0].set_xlabel('실제 피로도')
axes[0].set_ylabel('예측 피로도')
axes[0].set_title(f'예측값 vs 실제값 (R²={r2:.3f})')
axes[0].legend()

# 잔차 분포
axes[1].hist(residuals, bins=40, color='#10b981', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title(f'잔차 분포 (MAE={mae:.3f})')
axes[1].set_xlabel('잔차 (실제 - 예측)')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 모델 내보내기

학습된 모델을 `fatigue_ml_model.joblib`로 저장합니다.  
이 파일을 `millog-fatigue/models/` 디렉토리에 업로드하면 FastAPI에서 자동 로드합니다.

In [ ]:
import os
import json

os.makedirs('models', exist_ok=True)

# 모델 저장
MODEL_PATH = 'models/fatigue_ml_model.joblib'
joblib.dump(pipeline, MODEL_PATH)

# 메타데이터 저장
metadata = {
    'version': '1.0',
    'algorithm': 'Ridge Regression (RidgeCV)',
    'optimal_alpha': float(ridge_model.alpha_),
    'cv_mae': float(cv_mae.mean()),
    'cv_mae_std': float(cv_mae.std()),
    'cv_r2': float(cv_r2.mean()),
    'n_samples': len(df),
    'features_numerical': NUMERICAL_FEATURES,
    'features_categorical': CATEGORICAL_FEATURES,
    'target': 'reported_fatigue (0~10 사용자 자기보고)',
    'note': 'Phase 1: 합성 데이터 학습. Phase 2: 실제 500건+ 데이터로 재학습 필요.',
}

with open('models/model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

model_size = os.path.getsize(MODEL_PATH) / 1024
print(f'✅ 모델 저장 완료: {MODEL_PATH} ({model_size:.1f} KB)')
print(f'📊 메타데이터: models/model_metadata.json')
print()
print('다음 단계:')
print('  1. models/ 폴더를 millog-fatigue/models/ 에 업로드')
print('  2. millog-fatigue/fatigue/predictor.py 에 ML 모델 로드 코드 추가')
print('  3. /calculate 엔드포인트에서 규칙 기반 + ML 앙상블 적용')

## 8. FastAPI 통합 코드 (참고)

아래 코드를 `millog-fatigue/fatigue/predictor.py`에 추가하여 ML 모델을 통합하세요.

In [ ]:
INTEGRATION_CODE = '''
# millog-fatigue/fatigue/predictor.py 에 추가할 코드

import joblib
import pandas as pd
import os

MODEL_PATH = os.path.join(os.path.dirname(__file__), '..', 'models', 'fatigue_ml_model.joblib')
_ml_pipeline = None

def _load_model():
    global _ml_pipeline
    if _ml_pipeline is None and os.path.exists(MODEL_PATH):
        _ml_pipeline = joblib.load(MODEL_PATH)
    return _ml_pipeline

def predict_fatigue_ml(duty_type: str, duty_hours: float, time_zone: str,
                       consecutive_days: int, sleep_hours: float,
                       sleep_type: str, interruptions: int,
                       rank: str = "일병", branch: str = "육군") -> float | None:
    """ML 모델로 피로도 예측. 모델 없으면 None 반환 (규칙 기반 폴백용)"""
    model = _load_model()
    if model is None:
        return None

    DUTY_FATIGUE_INDEX = {
        "gop_gp": 2.63, "guard_duty": 2.45, "field_training": 2.33,
        "guard_post": 2.25, "night_watch": 2.16, "cctv_surveillance": 2.00,
        "indoor_training": 0.80, "admin_work": 0.50, "none": 0.00,
    }
    TIME_WEIGHT = {"day": 0.80, "evening": 0.90, "night": 1.00, "dawn": 1.15}
    SLEEP_QUALITY = {"continuous_6plus": 1.00, "split": 0.53, "under_4": 0.35}

    di = DUTY_FATIGUE_INDEX.get(duty_type, 0.0)
    ti = TIME_WEIGHT.get(time_zone, 0.80)
    c = min(1.0 + 0.15 * (consecutive_days - 1), 2.0)
    quality = SLEEP_QUALITY.get(sleep_type, 0.53)

    row = pd.DataFrame([{
        "duty_type": duty_type,
        "time_zone": time_zone,
        "sleep_type": sleep_type,
        "rank": rank,
        "branch": branch,
        "duty_hours": duty_hours,
        "consecutive_days": consecutive_days,
        "sleep_hours": sleep_hours,
        "interruptions": interruptions,
        "duty_fatigue": di * ti * duty_hours,
        "consecutive_factor": c,
        "sleep_recovery": sleep_hours * quality,
        "sleep_penalty": interruptions * 3.0,
        "is_night_dawn": float(time_zone in ["night", "dawn"]),
        "sleep_deficit": float(sleep_hours < 6),
    }])

    pred = model.predict(row)[0]
    return float(max(0.0, min(10.0, pred)))


# api/routes.py의 /calculate 엔드포인트에서:
# rule_score = calculate_fatigue(schedule).fatigue_score
# ml_score = predict_fatigue_ml(...)  # ML 예측
# final_score = (0.5 * rule_score + 0.5 * ml_score) if ml_score else rule_score  # 앙상블
'''

print(INTEGRATION_CODE)

## 9. 데이터 수집 API 명세 (Phase 1 → Phase 2 전환)

실제 서비스에서 데이터를 수집하려면 아래 엔드포인트를 FastAPI에 추가하세요.

In [ ]:
DATA_COLLECTION_API = '''
# millog-fatigue/api/routes.py 에 추가할 데이터 수집 엔드포인트

from fastapi import APIRouter
from pydantic import BaseModel
from pathlib import Path
import json
from datetime import datetime

LOG_FILE = Path("fatigue_logs.jsonl")

class FatigueLog(BaseModel):
    """사용자 자기보고 피로도 로그"""
    duty_type: str
    duty_hours: float
    time_zone: str              # day / evening / night / dawn
    consecutive_days: int
    sleep_hours: float
    sleep_type: str             # continuous_6plus / split / under_4
    interruptions: int
    rank: str                   # 이병 / 일병 / 상병 / 병장
    branch: str                 # 육군 / 해군 / 공군 / 해병대
    rule_fatigue: float         # 규칙 기반 계산값
    reported_fatigue: float     # 사용자 자기보고 피로도 (0~10, Likert 5점 × 2)

@router.post("/fatigue/log")
async def log_fatigue(log: FatigueLog):
    """피로도 계산 결과 + 사용자 보고값 저장"""
    entry = log.model_dump()
    entry["timestamp"] = datetime.utcnow().isoformat()
    with LOG_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\\n")
    return {"status": "logged"}

@router.get("/fatigue/logs/count")
async def get_log_count():
    """수집된 데이터 건수 확인"""
    if not LOG_FILE.exists():
        return {"count": 0, "ready_for_ml": False}
    count = sum(1 for _ in LOG_FILE.open())
    return {"count": count, "ready_for_ml": count >= 500}
'''

print(DATA_COLLECTION_API)

## 10. 다음 단계 로드맵

| 단계 | 조건 | 작업 |
|---|---|---|
| **Phase 1 (현재)** | 합성 데이터 | 파이프라인 완성, `/fatigue/log` API 추가 |
| **Phase 2** | 실제 데이터 500건+ | 이 노트북에서 재학습, 모델 배포 |
| **Phase 3** | 실제 데이터 5000건+ | LSTM 시계열 모델 (연속 근무 패턴) |

### Phase 2 재학습 체크리스트
1. `fatigue_logs.jsonl` 다운로드 (서버에서 `GET /fatigue/logs/export`)
2. Cell 1의 합성 데이터 코드를 실제 데이터 로드로 교체
3. 전체 노트북 재실행
4. `models/fatigue_ml_model.joblib` → `millog-fatigue/models/`에 업로드
5. Render 재배포